# Price tiers and customer identification

Descriptive price-ratio distributions and within-product elasticity estimates by CustomerID status.

## 1. Quantity vs relative price

Reproduce the product-relative price check on products with at least 20 clean lines. Price ratios use each product's median UnitPrice across all its clean lines. Within each customer group, bucket shares use that group's qualifying-line total as the denominator.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

project_root_candidates = (Path.cwd(), *Path.cwd().parents)
project_root = next(
    (candidate for candidate in project_root_candidates if (candidate / "src" / "clean.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the project root from the current directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.clean import load_clean
from src.features import build_product_week
from src.models import fit_within_product

clean = load_clean(verbose=False)


In [2]:
product_line_counts = clean.groupby("StockCode").size()
qualifying_products = product_line_counts.index[product_line_counts.ge(20)]
product_median_price = clean.groupby("StockCode")["UnitPrice"].transform("median")
clean_with_price_ratio = clean.assign(
    price_ratio=clean["UnitPrice"] / product_median_price
)
section1_lines = clean_with_price_ratio.loc[
    clean_with_price_ratio["StockCode"].isin(qualifying_products)
].copy()
section1_lines["quantity_bucket"] = pd.cut(
    section1_lines["Quantity"],
    bins=[0, 2, 5, 11, 23, np.inf],
    labels=["1-2", "3-5", "6-11", "12-23", "24+"],
    right=True,
)

assert len(section1_lines) == 517_559
assert int(section1_lines["CustomerID"].isna().sum()) == 128_801
assert int(section1_lines["CustomerID"].notna().sum()) == 388_758
print(
    "Qualifying-line assertions passed: 517,559 total, "
    "128,801 missing CustomerID, 388,758 present."
)

cohorts = {
    "All lines": section1_lines,
    "CustomerID missing": section1_lines.loc[section1_lines["CustomerID"].isna()],
    "CustomerID present": section1_lines.loc[section1_lines["CustomerID"].notna()],
}
price_tier_rows = []
for cohort_name, cohort_data in cohorts.items():
    cohort_total_lines = len(cohort_data)
    bucket_summary = cohort_data.groupby(
        "quantity_bucket", observed=False
    )["price_ratio"].agg(
        lines="size",
        median_price_ratio="median",
        p25_price_ratio=lambda values: values.quantile(0.25),
        p75_price_ratio=lambda values: values.quantile(0.75),
        share_price_ratio_lt_1=lambda values: 100 * values.lt(1).mean(),
    )
    bucket_summary["share_of_cohort_lines_pct"] = (
        100 * bucket_summary["lines"] / cohort_total_lines
    )
    bucket_summary["cohort_total_lines"] = cohort_total_lines
    bucket_summary = bucket_summary.reset_index()
    bucket_summary.insert(0, "customer_group", cohort_name)
    price_tier_rows.append(bucket_summary)

price_tier_summary = pd.concat(price_tier_rows, ignore_index=True)
price_tier_summary = price_tier_summary[
    [
        "customer_group",
        "quantity_bucket",
        "lines",
        "share_of_cohort_lines_pct",
        "median_price_ratio",
        "p25_price_ratio",
        "p75_price_ratio",
        "share_price_ratio_lt_1",
        "cohort_total_lines",
    ]
]
assert price_tier_summary.groupby("customer_group")["lines"].sum().to_dict() == {
    "All lines": 517_559,
    "CustomerID missing": 128_801,
    "CustomerID present": 388_758,
}
print("Quantity-bucket price ratios (shares use each customer group's own total):")
display(price_tier_summary)


Qualifying-line assertions passed: 517,559 total, 128,801 missing CustomerID, 388,758 present.


Quantity-bucket price ratios (shares use each customer group's own total):


,customer_group,quantity_bucket,lines,share_of_cohort_lines_pct,median_price_ratio,p25_price_ratio,p75_price_ratio,share_price_ratio_lt_1,cohort_total_lines
0,All lines,1-2,221926,42.879363,1.000000,1.000000,1.966667,4.580356,517559
1,All lines,3-5,85591,16.537438,1.000000,1.000000,1.000000,6.101109,517559
2,All lines,6-11,80800,15.611747,1.000000,1.000000,1.000000,8.226485,517559
3,All lines,12-23,73967,14.291511,1.000000,1.000000,1.000000,9.719199,517559
4,All lines,24+,55275,10.679942,1.000000,0.864407,1.000000,33.794663,517559
5,CustomerID missing,1-2,95032,73.782036,1.968000,1.777358,2.013793,4.642647,128801
6,CustomerID missing,3-5,19775,15.353142,1.962712,1.000000,1.999216,10.098609,128801
7,CustomerID missing,6-11,8665,6.727432,1.945098,1.000000,1.997590,13.179458,128801
8,CustomerID missing,12-23,3448,2.676998,1.920362,1.000000,2.148235,11.948956,128801
9,CustomerID missing,24+,1881,1.460392,2.000000,1.000000,2.862069,8.771930,128801


## 2. Elasticity by customer type

Build product-week samples separately for all, identified-customer, and missing-CustomerID lines. Drop the two flagged partial weeks and retain products with at least 10 weeks and at least two distinct weekly modal prices.

In [3]:
partial_week_starts = pd.to_datetime(["2010-11-29", "2011-12-05"])

def build_eligible_weekly_sample(lines):
    weekly = build_product_week(lines)
    weekly = weekly.loc[~weekly["week"].isin(partial_week_starts)].copy()
    weekly["log_price"] = np.log(weekly["price_mode"])
    weekly["log_units"] = np.log(weekly["units"])
    weekly["log_invoices"] = np.log(weekly["invoices"])
    weekly["week_id"] = weekly["week"].astype(str)
    log_columns = ["log_price", "log_units", "log_invoices"]
    assert weekly[log_columns].notna().all().all()
    assert np.isfinite(weekly[log_columns].to_numpy()).all()

    eligibility = weekly.groupby("StockCode").agg(
        weeks=("week", "nunique"),
        distinct_prices=("price_mode", "nunique"),
    )
    eligible_products = eligibility.index[
        eligibility["weeks"].ge(10) & eligibility["distinct_prices"].ge(2)
    ]
    sample = weekly.loc[weekly["StockCode"].isin(eligible_products)].copy()
    assert sample.groupby("StockCode")["week"].nunique().ge(10).all()
    assert sample.groupby("StockCode")["price_mode"].nunique().ge(2).all()
    return sample

customer_groups = {
    "All": clean,
    "Identified": clean.loc[clean["CustomerID"].notna()].copy(),
    "Missing": clean.loc[clean["CustomerID"].isna()].copy(),
}
weekly_samples = {
    group_name: build_eligible_weekly_sample(group_data)
    for group_name, group_data in customer_groups.items()
}

model_rows = []
for group_name, sample in weekly_samples.items():
    for outcome in ("log_invoices", "log_units"):
        fitted = fit_within_product(sample, outcome)
        ci_lower, ci_upper = fitted.conf_int().loc["log_price"]
        model_rows.append(
            {
                "group": group_name,
                "outcome": outcome,
                "coefficient": fitted.params["log_price"],
                "clustered SE": fitted.bse["log_price"],
                "95% CI lower": ci_lower,
                "95% CI upper": ci_upper,
                "observations": int(fitted.nobs),
                "products": sample["StockCode"].nunique(),
            }
        )

elasticity_results = pd.DataFrame(model_rows)
assert not elasticity_results.isna().any().any()
all_invoice_result = elasticity_results.loc[
    elasticity_results["group"].eq("All")
    & elasticity_results["outcome"].eq("log_invoices")
].iloc[0]
assert abs(all_invoice_result["coefficient"] - (-0.397409)) < 1e-6
assert int(all_invoice_result["observations"]) == 83_377
assert int(all_invoice_result["products"]) == 2_523
print(
    "All-clean validation passed: log_invoices coefficient matches -0.397409 "
    "within 1e-6, with 83,377 observations and 2,523 products."
)
print("Within-product OLS results (clustered by product):")
display(elasticity_results)


All-clean validation passed: log_invoices coefficient matches -0.397409 within 1e-6, with 83,377 observations and 2,523 products.
Within-product OLS results (clustered by product):


,group,outcome,coefficient,clustered SE,95% CI lower,95% CI upper,observations,products
0,All,log_invoices,-0.397409,0.014191,-0.425223,-0.369594,83377,2523
1,All,log_units,-1.502479,0.024650,-1.550792,-1.454167,83377,2523
2,Identified,log_invoices,-0.493074,0.024273,-0.540648,-0.445500,50237,1597
3,Identified,log_units,-1.703228,0.040197,-1.782013,-1.624443,50237,1597
4,Missing,log_invoices,-0.494630,0.022055,-0.537857,-0.451403,45449,1851
5,Missing,log_units,-1.324769,0.032200,-1.387880,-1.261657,45449,1851


## 3. Interpretation

**Observation.** The all-clean `log_invoices` coefficient is -0.397409; the identified-customer estimate is -0.493074 (95% CI -0.540648 to -0.445500), and the missing-CustomerID estimate is -0.494630 (95% CI -0.537857 to -0.451403).

**Meaning.** These estimates compare within-product changes in weekly modal price with changes in recorded weekly invoices, accounting for common week effects. Whether missing CustomerID represents a retail tier is a hypothesis, not an observed customer-type label.

**Decision rule.** Treat an invoice coefficient with absolute value below 0.05 as near zero. Both customer groups have negative log_invoices coefficients that are not near zero; this customer-type split supports the negative association in the main model. The main model is unchanged.

**What this does not mean.** The split is not causal evidence and does not prove that records with missing CustomerID are retail sales. It is a customer-ID-status sensitivity check, not a reason to change the main model.